# Confidence interval of survival data 
## Introduction
## Survival analysis
### Definitions
### Survival data


In [ ]:
import pandas as pd

# Example of the survival data
durations       = [4.07, 6.54, 1.39, 6.17, 5.89, 4.76, 3.67]
event_observed  = [1   , 0   , 1   , 0   , 1   , 1   , 0   ]

# Import the data in a DataFrame
data = pd.DataFrame({
    'time': durations,
    'event': event_observed})

# Sorted durations
data = data.sort_values(by='time', ignore_index=True)

print("Survival data (sorted by durations):")
print(data)

### Survival fraction


In [ ]:
n0 = len(data)           # Initial number at risk

print("Survival fractions:")

# At t1
t1 = data.iloc[0, 0]     # Time at t1
n1 = n0                  # Number at risk *just before* t1
d1 = data.iloc[0, 1]     # Number of events at t1 (first row)
c1 = 1 - d1              # Number of censored at t1
S1 = (1 - d1 / n1)       # Survival fraction at t1
print(f"{t1=} → {S1=:.4f}")
# Notice the self-documenting expressions in the f-string ;-)

# At t2
t2 = data.iloc[1, 0]     # Time at t2
n2 = n1 - d1 - c1        # Number at risk *just before* t2
d2 = data.iloc[1, 1]     # Number of events at t2 (d2=0)
c2 = 1 - d2              # Number of censored at t2
S2 = S1 * (1 - d2 / n2)  # Survival fraction (same as S1)
print(f"{t2=} → {S2=:.4f}")

# At t3
t3 = data.iloc[2, 0]     # Time at t3
n3 = n2 - d2 - c2        # Number at risk *just before* t3
d3 = data.iloc[2, 1]     # Number of events at t3
c3 = 1 - d3              # Number of censored at t3
S3 = S2 * (1 - d3 / n3)  # Survival fraction at t3
print(f"{t3=} → {S3=:.4f}")

# At t4
t4 = data.iloc[3, 0]     # Time at t4
n4 = n3 - d3 - c3        # Number at risk *just before* t4
d4 = data.iloc[3, 1]     # Number of events at t4
c4 = 1 - d4              # Number of censored at t4
S4 = S3 * (1 - d4 / n4)  # Survival fraction at t4
print(f"{t4=} → {S4=:.4f}")

### Survival function


In [ ]:
import numpy as np

def custom_kaplan_meier(durations, event_observed):
    """
    Estimates the survival function using the Kaplan-Meier method

    Args:
        durations: a numpy array of event times
        event_observed: a numpy array of indicators
            (1 for event, 0 for censoring) 

    Returns:
        A tuple containing three numpy arrays:
            - survival_prob_sorted: estimated survival probabilities sorted
            - events_sorted: unique event (sorted)
            - durations_sorted: unique event times (sorted)
    """

    # Sort durations and event_observed by durations
    sort_indices = np.argsort(durations)
    durations_sorted = np.array(durations)[sort_indices]
    events_sorted = np.array(event_observed)[sort_indices]

    # Initialize variables
    n_at_risk = len(durations_sorted)  # initial number at risk
    survival_prob_sorted = np.ones_like(durations_sorted)

    for i in range(len(durations_sorted)):
        if events_sorted[i] == 1:  # if event occurred
            survival_prob_sorted[i:] *= (1 - 1/n_at_risk)
        n_at_risk -= 1  # decrease number at risk

    return survival_prob_sorted, events_sorted, durations_sorted

In [ ]:
km_prob, km_events, km_durations = custom_kaplan_meier(
    data['time'], data['event'])

print("Kaplan-Meier probabilities (custom function):")
# Print survival probabilities at each event time
for i in range(len(km_durations)):
    print(f"Time: {km_durations[i]} → S(t) = {km_prob[i]:.5f}")

## Confidence intervals for the survival function
### The basis
### Variance of the Kaplan-Meier estimator


### Step-by-step implementation


In [ ]:
# Greenwood's variance component at t1
# d1, n1 and S1 were calculated in a previous snippet
variance_component_t1 = d1 / (n1 * (n1 - d1))

# Cumulative sum of variance components up to t1 (only one term so far)
cumulative_variance_t1 = variance_component_t1

# Standard error at t1
se1 = S1 * np.sqrt(cumulative_variance_t1)

# Calculate 95% confidence interval limits (using z* = 1.96)
z_crit = 1.96
lower_ci_t1 = S1 - z_crit * se1
upper_ci_t1 = S1 + z_crit * se1

# Print results with clear labels and formatting
print(f"At time {t1}:")
print(f" Kaplan-Meier estimate (S) = {S1:.4f}")
print(f" Greenwood variance component = {variance_component_t1:.3f}")
print(f" Cumulative variance = {cumulative_variance_t1:.3f}")
print(f" Standard error (s) = {se1:.4f}")
print(f" 95% confidence interval (plain): [{lower_ci_t1:.3f}, \
{upper_ci_t1:.3f}]")

In [ ]:
# Greenwood's variance component at t2
#  Since d2 = 0, the variance component at t2 is 0.  We *still* calculate it
#  for completeness and to demonstrate the general formula.
variance_component_t2 = d2 / (n2 * (n2 - d2))

# Cumulative sum of variance components up to t2
cumulative_variance_t2 = cumulative_variance_t1 + variance_component_t2

# Standard error at t2
se2 = S2 * np.sqrt(cumulative_variance_t2)

# Calculate 95% confidence interval limits
lower_ci_t2 = S2 - z_crit * se2
upper_ci_t2 = S2 + z_crit * se2

# Print results for t2
print(f"At time {t2}:")
print(f" Kaplan-Meier estimate (S) = {S2:.4f}")
print(f" Greenwood variance component = {variance_component_t2:.3f}")
print(f" Cumulative variance = {cumulative_variance_t2:.3f}")
print(f" Standard error (s) = {se2:.4f}")
print(f" 95% confidence interval (plain): \
[{lower_ci_t2:.3f}, {upper_ci_t2:.3f}]")

In [ ]:
# Greenwood's variance component at t3 and t4
variance_component_t3 = d3 / (n3 * (n3 - d3))
variance_component_t4 = d4 / (n4 * (n4 - d4))

# Cumulative sum of variance components up to t3 and t4
cumulative_variance_t3 = cumulative_variance_t2 + variance_component_t3
cumulative_variance_t4 = cumulative_variance_t3 + variance_component_t4

# Standard errors
se3 = S3 * np.sqrt(cumulative_variance_t3)
se4 = S4 * np.sqrt(cumulative_variance_t4)

# 95% confidence interval limits
lower_ci_t3 = S3 - z_crit * se3
upper_ci_t3 = S3 + z_crit * se3

lower_ci_t4 = S4 - z_crit * se4
upper_ci_t4 = S4 + z_crit * se4

# Print results
print(f"At time {t3}:")
print(f" Kaplan-Meier estimate (S) = {S3:.4f}")
print(f" Greenwood variance component = {variance_component_t3:.3f}")
print(f" Cumulative variance = {cumulative_variance_t3:.3f}")
print(f" Standard error (s) = {se3:.4f}")
print(f" 95% confidence interval (plain): [{lower_ci_t3:.3f}, \
{upper_ci_t3:.3f}]")
print()
print(f"At time {t4}:")
print(f" Kaplan-Meier estimate (S) = {S4:.4f}")
print(f" Greenwood variance component = {variance_component_t4:.3f}")
print(f" Cumulative variance = {cumulative_variance_t4:.3f}")
print(f" Standard error (s) = {se4:.4f}")
print(f" 95% confidence interval (plain): [{lower_ci_t4:.3f}, \
{upper_ci_t4:.3f}]")

### Log survival


In [ ]:
# Log-log transformation (t1)
log_survival_increment_t1 = np.log(1 - d1/n1)
cumulative_log_survival_t1 = log_survival_increment_t1  # Denominator

# Standard error using Greenwood component
se_v1 = np.sqrt(cumulative_variance_t1 / cumulative_log_survival_t1**2)

lower_ci_loglog_t1 = S1**(np.exp( z_crit * se_v1))
upper_ci_loglog_t1 = S1**(np.exp(-z_crit * se_v1))

print(f"At time {t1}:")
print(f" Log-survival increment = {log_survival_increment_t1:.3f}")
print(f" Cumulative log-survival = {cumulative_log_survival_t1:.3f}")
print(f" Log-log standard error = {se_v1:.4f}")
print(f" 95% CI (back-transformed): [{lower_ci_loglog_t1:.4f}, \
{upper_ci_loglog_t1:.4f}]")

In [ ]:
# Log-log transformation (t2)
log_survival_increment_t2 = np.log(1 - d2/n2)
cumulative_log_survival_t2 = (
    cumulative_log_survival_t1 + log_survival_increment_t2)

# Standard error using Greenwood component
se_v2 = np.sqrt(cumulative_variance_t2 / cumulative_log_survival_t2**2)

lower_ci_loglog_t2 = S2**(np.exp( z_crit * se_v2))
upper_ci_loglog_t2 = S2**(np.exp(-z_crit * se_v2))

print(f"At time {t2}:")
print(f" Log-survival increment = {log_survival_increment_t2:.3f}")
print(f" Cumulative log-survival = {cumulative_log_survival_t2:.3f}")
print(f" Log-log standard error = {se_v2:.4f}")
print(f" 95% CI (back-transformed): [{lower_ci_loglog_t2:.4f}, \
{upper_ci_loglog_t2:.4f}]")

In [ ]:
# Log-log transformation (t3 and t4)
log_survival_increment_t3 = np.log(1 - d3/n3)
cumulative_log_survival_t3 = (
    cumulative_log_survival_t2 + log_survival_increment_t3)

log_survival_increment_t4 = np.log(1 - d4/n4)
cumulative_log_survival_t4 = (
    cumulative_log_survival_t3 + log_survival_increment_t4)

# Standard errors and CI limits using Greenwood component
se_v3 = np.sqrt(cumulative_variance_t3 / cumulative_log_survival_t3**2)
se_v4 = np.sqrt(cumulative_variance_t4 / cumulative_log_survival_t4**2)

lower_ci_loglog_t3 = S3**(np.exp( z_crit * se_v3))
upper_ci_loglog_t3 = S3**(np.exp(-z_crit * se_v3))

lower_ci_loglog_t4 = S4**(np.exp( z_crit * se_v4))
upper_ci_loglog_t4 = S4**(np.exp(-z_crit * se_v4))

print(f"At time {t3}:")
print(f" Log-survival increment = {log_survival_increment_t3:.3f}")
print(f" Cumulative log-survival = {cumulative_log_survival_t3:.3f}")
print(f" Log-log standard error = {se_v3:.4f}")
print(f" 95% CI (back-transformed): [{lower_ci_loglog_t3:.4f}, \
{upper_ci_loglog_t3:.4f}]")
print()
print(f"At time {t4}:")
print(f" Log-survival increment = {log_survival_increment_t4:.3f}")
print(f" Cumulative log-survival = {cumulative_log_survival_t4:.3f}")
print(f" Log-log standard error = {se_v4:.4f}")
print(f" 95% CI (back-transformed): [{lower_ci_loglog_t4:.4f}, \
{upper_ci_loglog_t4:.4f}]")

### Automation with scripts


In [ ]:
from scipy.stats import norm

def custom_greenwood_confidence_interval(
        survival_prob_sorted, event_indicators_sorted,
        alpha=.05, method="log-log"):
    """
    Calculates the confidence intervals, plain or complementary log-log, for 
    Kaplan-Meier estimates, and assuming pre-sorted data, as obtained with
    custom_kaplan_meier, using the Greenwood's formula discussed earlier.

    Args:
        survival_prob_sorted: a (numpy) array of estimated survival 
            probabilities at each time-point (assumed to be sorted).
        event_indicators: a (numpy) array of indicators (1 for event, 
            0 for censoring) at each time-point (assumed to be sorted).
        alpha: significance level (e.g., 0.05 for 95% confidence interval)
        method: if not "log-log" then the function return the "plain" 
            or "linear" confidence intervals

    Returns:
        A tuple containing two numpy arrays:
            standard error using the given method (log-log by default) 
                and alpha (95% confidence by default).
            lower and upper bounds of the confidence interval using the 
                given method and alpha.
    """

    # Use inverse normal cumulative distribution for z-score, and not
    # the approximation to 1.96
    z = norm.ppf(1 - alpha/2)  # It's also a two-tailed test...

    # We first calculate the term in the sum for the variance
    # di are indeed the elements of the array `event_indicators_sorted`
    d = event_indicators_sorted

    # We get the number of individual at risk remaining at each time-point
    n = len(d) + 1 - np.cumsum(np.ones_like(d))
    
    # We then compute the cumulative sums of the terms
    sum_e = np.cumsum(d / (n * (n - d)))

    # The cumulative sums of the log of the term for the log-log method
    sum_l = np.cumsum(np.log((n - d) / n))
    
    # If the method is not "log-log", then return the "plain" SE and CI
    if method != "log-log":
        return (
            SE:=survival_prob_sorted * np.sqrt(sum_e),
            np.array(
                [survival_prob_sorted - z*SE, survival_prob_sorted + z*SE]))
    else:
        return (
            SE:=np.sqrt(sum_e / sum_l**2),
            np.array([
                survival_prob_sorted**np.exp(1.96*SE),
                survival_prob_sorted**np.exp(-1.96*SE)]))

In [ ]:
# Let enter the KM estimates obtained previously
SE_KM, ci_KM = custom_greenwood_confidence_interval(km_prob, km_events)

print("95% confidence interval (log-log; custom function):")
print(ci_KM.round(4))

## The lifelines library


In [ ]:
from lifelines import KaplanMeierFitter

# Create a fitter instance
kmf = KaplanMeierFitter()

# Fit the data to the model
kmf.fit(
    durations=durations,
    event_observed=event_observed,)

### Estimates and statistics


In [ ]:
print("Survival function estimate:")
print(kmf.survival_function_)

In [ ]:
print("Median survival time:", kmf.median_survival_time_)

### Confidences intervals


In [ ]:
print("Confidence intervals for each time-point:")
print(kmf.confidence_interval_)

### More utilities


In [ ]:
from lifelines.utils import survival_table_from_events

# Create the survival table
table = survival_table_from_events(durations, event_observed)

print("Survival table:")
print(table)

In [ ]:
from lifelines.utils import survival_events_from_table

# Need to create new columns in order to prepare a lifelines table
data['observed'] = data['event'] == 1
data['censored'] = data['event'] == 0

# Transforming survival-table data into lifelines format
T, E, W = survival_events_from_table(
    data.set_index('time'),
    observed_deaths_col='observed',
    censored_col='censored')

print("Durations of observation:")
print(T)

### Plotting the survival function


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(4.5, 3.5))

ax = kmf.plot_survival_function(  # Same as kmf.plot()
    show_censors=True,   # Place markers at censorship events
    ci_show=True,        # Show confidence intervals; default is True
    legend=False,        # Don't display the legend
    lw=3, c='red',       # Customization of the line
    at_risk_counts=True, # Add summary tables under the plot
)
ax.set_xlabel("Years")
ax.set_ylabel("Suvival probability")
ax.set_title("Kaplan-Meier survival curve");

## Conclusion
